# Message Authentication Code (MAC) tag

**Feladat**: Alice és Bob korábban megegyezett egy titkos kulcsban. Alice átküldött egy titkosított üzenetet Bobnak. Bob biztos akar lenni abban, hogy az üzenet valóban Alice-tól jött és nem valaki mástól. Hogyan *authentikálja* Bob Alice titkosított üzenetét?

**Definíció**: Message Authentication Code
- $\mathtt{Gen}$: titkos $k$ kulcsot generál egyenletes eloszlás szerint
- $\mathtt{Mac}$: készít egy $t$ *tag*-et az $m$ üzenetből, amit a titkosított üzenethez csatolunk: $t \leftarrow \mathtt{Mac}(k, m)$
- $\mathtt{Vrfy}$: adott $k$ titkos kulcs, $m$ üzenet és $t$ tag. A kimenet $b=1$ ha a tag jó, egyébként $b=0$: $b := \mathtt{Vrfy}(k, m, t)$

Követelmény: $\mathtt{Vrfy}(k, m, \mathtt{Mac}(k, m)) = 1$

**Definíció**: Egy MAC tag 

## CBC-MAC

![CBC-MAC](./cbc-mac.png)

In [3]:
from des import *
from utils import *

import secrets
from typing import Optional

In [4]:
class DES_CBC_MAC(DES):
    def __init__(self, key, block_size: int = 8, iv: Optional[bytes] = None) -> None:
        super().__init__(key, block_size)
        self.iv = bytes(self.block_size)
        
    def mac(self, message: bytes) -> bytes:
        tag = self.iv
        
        for b in range(0, len(message), self.block_size):
            msg_block = message[b:b+self.block_size]
            inp = xor_strings(tag[-self.block_size:], msg_block)
            tag += self._encryption(inp, True)
        
        return tag[-self.block_size:]
        
    def vrfy(self, message: bytes, tag: bytes) -> bool:
        return tag == self.mac(message)

In [5]:
message = b'cryptography0123'

In [6]:
mac_master = DES_CBC_MAC(b'animator')
tag_master = mac_master.mac(message)
print(tag_master)
print(mac_master.vrfy(message, tag_master))

b'\x9b\xb9\xf7C\x85\xca\xf4\xe1'
True


**Kérdés**: Mi lehet a gond a fenti konstrukcióval?

Az első probléma demonstrációja:

Az IV nem lehet random


In [7]:
def invert_byte(b):
    return bytes.fromhex(f'{b ^ 0xff:02x}')

In [8]:
message_ = invert_byte(message[0]) + message[1:]
iv_ = invert_byte(mac_master.iv[0]) + mac_master.iv[1:]
print(message_)
print(iv_)

b'\x9cryptography0123'
b'\xff\x00\x00\x00\x00\x00\x00\x00'


In [9]:
mac2 = DES_CBC_MAC(b'animator', iv=iv_)
tag_ = mac2.mac(message_)
print(tag_master)
print(tag_)

b'\x9b\xb9\xf7C\x85\xca\xf4\xe1'
b'\xd3\xf5j\x90~;\x15W'


A második probléma demonstrációja:

A titkositashoz es tag kesziteshez a kulcs ugyanaz


In [10]:
des = DES_CBC(b'reinvent', iv=bytes(8))
des_mac = DES_CBC_MAC(b'reinvent')
ct = des.encrypt(message, False)
ct_tag = des_mac.mac(message)
ct, ct_tag

(b"\x00\x00\x00\x00\x00\x00\x00\x00\xbb\xe2\xcf`\xa7\xb2\x05\x9aE\xdd\xf6\x98\xea\xa2<'",
 b"E\xdd\xf6\x98\xea\xa2<'")

In [11]:
ct_ = ct[:8] + secrets.token_bytes(8) + ct[16:]
ct_

b"\x00\x00\x00\x00\x00\x00\x00\x00h\xc1s!\x90\xc1g]E\xdd\xf6\x98\xea\xa2<'"

In [12]:
decrypted = des.decrypt(ct_, False)

In [13]:
des_mac.vrfy(decrypted, ct_tag)

True

**Kérdés**: Legyen $m$ egy pontosan egy blokk hosszú üzenet. gy kétszer olyan hosszú üzenetnek ugyanaz lehet a tag-je. Hogyan?

**Megoldás**: Legyen $m$ egy egy-blokk hosszúságú üzenet, aminek a tag-je $t = F(k, m)$. Legyen $m' = m\;||\;(m \oplus t)$. Ekkor $t_1 = F(k, m) = t$ és $t_2 = F(k, t \oplus (t \oplus m)) = F(k, t \oplus t \oplus m) = F(k, m) = t$, azaz $t_1 = t_2$

Length extension attack


In [14]:
message = b'humanoid'

In [17]:
mac = DES_CBC_MAC(secrets.token_bytes(8))
tag = mac.mac(message)
tag

b'\x86\xa2\x87\x00\xa5xk\xb4'

In [18]:
message_2 = message + xor_strings(message, tag)
message_2

b'humanoid\xee\xd7\xeaa\xcb\x17\x02\xd0'

In [19]:
mac.mac(message_2) == tag

True

In [20]:
len(message).to_bytes(1, 'big') + message

b'\x08humanoid'

Javítás lehet: tegyük az üzenet elé az üzenet hosszát, azaz $m' = |m| + m$ és ennek számoljuk ki a tag-ét

## Autentikált titkosítás

Legyenek $k_E$ és $k_M$ egymástól független kulcsok, legyen $m$ egy üzenet.

- *Encrypt-and-authenticate*: titkosítás és authentikáció egymástól független: $c \leftarrow \mathtt{Enc}(k_E, m) \quad t \leftarrow \mathtt{Mac}(k_M, m)$
    - Nem feltétlenül lesz CPA-biztonságos: ha CBC-MAC-et használunk, akkor ugyanaz az üzenet ugyanolyan tag-et kap
- *Authenticate-then-encrypt*: előbb a MAC tag-et számoljuk ki, majd a tag-et és az $m$ üzenetet együtt titkosítjuk: $t \leftarrow \mathtt{Mac}(k_M, m) \quad c \leftarrow \mathtt{Enc}(k_E, m||t)$
    - Visszafejtéskor két hiba lehet: `BadPaddingError` vagy érvénytelen tag
    - Ha a két exception megkülönböztethető, padding-oracle attack alkalmazható
    - IPsec támadása 2010-ben
    - SSL: megkülönböztethetetlen exception-k, de side-channel attack-al törhető volt
- *Encrypt-then-authenticate*: $c \leftarrow \mathtt{Enc}(k_E, m) \quad t \leftarrow \mathtt{Mac}(k_M, c)$
    - Egy titkos szöveget előbb verifikálunk, után fejtünk vissza
    - Padding-oracle attack-nak ellenáll

**Kérdés**: Miért kell, hogy $k_E$ és $k_M$ egymástól függetlenek legyenek?

**Megoldás**: Mert ha ugyanazt a kulcsot használjuk, visszakapjuk a plaintext-et:
- Legyen $\mathtt{Enc}(k, m) = F(k, m||r)$ és $\mathtt{Mac}(k, c) = F^{-1}(k, c)$, ahol $m \in \{0,1\}^{n/2}, r \in \{0,1\}^{n/2}$
- $\mathtt{Enc}(k, m) = F(k, m||r)$
- $\mathtt{Mac}(\mathtt{Enc}(k, m||r)) = F^{-1}(k, F(k, m||r)) = m||r$